In [ ]:
from datetime import date, datetime, timedelta

import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import numpy as np
import polars as pl
import spaceplot as sp

import tidal

sp.display('dark', retina=True)


In [ ]:
begin, end = '20260101', '20261231'
pred = tidal.noaa_tides(begin=begin, end=end, interval='5')
pred = tidal.normalize_amplitude(pred)


In [ ]:
dates = pred.unique(pl.col('t').dt.date())['t']
sun_times = tidal.get_sun_times(dates).sort('date')
sun_times.head()


In [ ]:
df = pred

day_tides = pl.DataFrame()
for daytime in ['sunrise', 'sunset']:
    time_tides = pl.DataFrame()
    for target in sun_times[daytime]:
        target = target.replace(tzinfo=None)
        res = df.with_columns(delta=(pl.col('t') - pl.lit(target)).abs()).sort('delta').head(1)

        time_tides = time_tides.vstack(res)

    time_tides = time_tides.with_columns(pl.lit(daytime).alias('daytime'))
    day_tides = day_tides.vstack(time_tides)

day_tides.head()


In [ ]:
df = (
    day_tides.with_columns(pl.col('t').dt.month().alias('month'))
    .with_columns(pl.col('t').dt.week().alias('week'))
    .with_columns(pl.col('t').dt.weekday().alias('weekday'))
    .with_columns(pl.col('t').dt.quarter().alias('quarter'))
    .with_columns(pl.col('t').dt.strftime('%A').alias('weekday_name'))
    .with_columns(pl.col('t').dt.strftime('%B').alias('month_name'))
    .with_columns(pl.col('t').dt.day().alias('day'))
)

weekdays = df.unique('weekday_name').sort('weekday')['weekday_name'].to_list()
df_arrayed = (
    df.sort('weekday')
    .pivot(on='weekday_name', values='amplitude', index=['month', 'month_name', 'week', 'daytime'])
    .sort('month')
    .sort('week')
)


In [ ]:
daytime = 'sunset'

axs = sp.montage_plot(
    12,
    n_cols=6,
    panel_size=(3, 3),
    layout='compressed',
    wspace=0.1,
    hspace=0.0,
    title=f'Tidal amplitude during {daytime.capitalize()}',
)

norm = mcolors.Normalize(vmin=-1, vmax=1)
for ax, mon in zip(axs, range(1, 13)):
    plot_df = df.filter(pl.col('month') == mon, pl.col('daytime') == daytime)

    radius = abs(plot_df['amplitude'] * 20)  # desired radius in points

    sc = ax.scatter(
        plot_df['weekday'],
        plot_df['week'],
        c=plot_df['amplitude'],
        norm=norm,
        s=radius**2,
        cmap='cmc.managua_r',
    )

    for row in plot_df.iter_rows(named=True):
        ax.text(row['weekday'], row['week'] + 0.5, row['day'], fontsize=8, ha='center', va='center', color='0.5')

    ax.set_title(f'{plot_df[0, "month_name"]}', fontsize=14, pad=8)

    times = (
        plot_df.group_by('week')
        .agg(pl.col('t').min())
        .with_columns(pl.col('t').dt.strftime('%H:%M').alias('date'))
        .sort('week')['date']
        .to_list()
    )

    sp.layout(
        ax,
        y_lims=(plot_df['week'].min() - 0.5, plot_df['week'].max() + 1),
        y_breaks=plot_df['week'].unique().to_list(),
        # y_ticklabels=times,
        y_tick_size=0,
    )

# after the plotting loop
cbar = axs.figure.colorbar(
    sc,
    ax=[axs[5], axs[11]],  # June and December
    location='right',
    shrink=0.5,
    fraction=1,
    pad=0.05,
)

axs.layout(x_ticklabels=[w[0:2] for w in weekdays], x_breaks=np.arange(1, len(weekdays) + 1))
sp.layout(axs[0:-6], x_ticks=False)
axs.invert_yaxis()

sp.show()
